This project analyzes an online retail sales dataset to uncover business insights that support data-driven decision-making. Using Python and Jupyter Notebook, the analysis evaluates sales performance, product trends, customer purchasing behavior, and geographic sales patterns through exploratory data analysis and visualization.

The project aims to answer key business questions such as:

Which products generate the highest revenue and sales volume?
Which countries and periods contribute most to overall sales?
What purchasing patterns drive business performance?
Which products should be prioritized based on their business impact?

The findings are presented through clear visualizations and actionable insights, demonstrating how data can be used to improve sales performance, optimize product strategy, and support informed business decisions.

In [1]:
#Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading Online Retail Datasert and returning first three rows

df_sales = pd.read_csv(r"C:\Users\Samiran\AI Engineering\Github\Online-Retail-Sales-and-RFM-Analysis\data\processed\sales_data.csv")
df_sales.head(3)

C:\Users\Samiran\AppData\Local\Temp\ipykernel_9176\586528082.py:3: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sales = pd.read_csv(r"C:\Users\Samiran\AI Engineering\Github\Online-Retail-Sales-and-RFM-Analysis\data\processed\sales_data.csv")


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


In [3]:
df_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 526052 entries, 0 to 526051
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    526052 non-null  object 
 1   StockCode    526052 non-null  str    
 2   Description  525460 non-null  str    
 3   Quantity     526052 non-null  int64  
 4   InvoiceDate  526052 non-null  str    
 5   UnitPrice    526052 non-null  float64
 6   CustomerID   392732 non-null  float64
 7   Country      526052 non-null  str    
dtypes: float64(2), int64(1), object(1), str(4)
memory usage: 32.1+ MB


In [4]:
df_sales['InvoiceDate']=pd.to_datetime(df_sales['InvoiceDate'])
df_sales['InvoiceDate'].dtype

dtype('<M8[us]')

In [5]:
df_sales.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,526052.000000,526052,526052.000000,392732.000000
mean,10.730911,2011-07-04 13:30:26.891600,3.913818,15287.734822
min,1.000000,2010-12-01 08:26:00,0.000000,12346.000000
25%,1.000000,2011-03-28 11:36:00,1.250000,13955.000000
50%,4.000000,2011-07-19 17:17:00,2.080000,15150.000000
75%,11.000000,2011-10-19 11:13:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,13541.330000,18287.000000
std,157.592136,NaN,36.053205,1713.567773


Feature Engineering:
We will create additional features to support sales, product and customer analysis

In [6]:
#creating revenue column
df_sales['Revenue'] = df_sales['Quantity']*df_sales['UnitPrice']

Next the date column details given, we would like to break it down into Year, Month, Day, Quarter and Hour so that we can analyse the times across which sales are distributed.
In addition, we would also like to see the weekend sales compared to weekday sales, so will create a weekend flag

In [7]:
df_sales['Year'] = df_sales['InvoiceDate'].dt.year
df_sales['Month'] = df_sales['InvoiceDate'].dt.month
df_sales['MonthName'] = df_sales['InvoiceDate'].dt.month_name()

#below codes gives an order to the months so that we have chronological order and not alphabetical order
months = ["January","February","March","April","May","June","July","August","September","October","November","December"]
df_sales['MonthName'] = pd.Categorical(df_sales['MonthName'], categories=months, ordered=True)

df_sales['Quarter'] = df_sales['InvoiceDate'].dt.quarter
df_sales['DayOfWeek'] = df_sales['InvoiceDate'].dt.day_name()
df_sales['IsWeekend'] = df_sales['InvoiceDate'].dt.weekday >= 5
df_sales['Hour'] = df_sales['InvoiceDate'].dt.hour

In case of invoice, we would like to have a look at the total invoice value, i.e the sum of all products sold as a single invoice. This would help us in understanding the order size

In [8]:
#Here we are creating a new column Invoice Value and adding it to the original df_sales 
invoice_value = (df_sales.groupby('InvoiceNo')['Revenue'].sum().rename('InvoiceValue'))
df_sales = df_sales.merge(invoice_value,on='InvoiceNo',how = 'left')

To enhance the analytical value of the dataset, additional features were derived from the existing transaction data. These engineered features enable analysis at both the transaction and time levels, supporting deeper business insights without altering the underlying data.
1. Revenue was calculated as the product of Quantity × Unit Price, providing the sales value for each transaction line and serving as the primary metric for revenue analysis.
2. Invoice Value was computed by aggregating the revenue of all items within an invoice, enabling order-level analysis such as average order value and customer purchasing patterns.
3. Invoice Date was converted to a datetime format to facilitate temporal analysis.
4. Year, Quarter, Month, and Month Name were extracted from the invoice date to support trend analysis across different time periods and identify seasonal sales patterns.
5. Day of Week was derived to analyze customer purchasing behavior across weekdays.
6. Hour was extracted to identify peak purchasing periods and understand transaction activity throughout the day.

These engineered features establish the foundation for the subsequent exploratory data analysis, allowing business performance to be evaluated across sales, products, customers, geography, and time dimensions.

In [9]:
#We will have a quick look at our data after doing the feature engineering
df_sales.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Year,Month,MonthName,Quarter,DayOfWeek,IsWeekend,Hour,InvoiceValue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,December,4,Wednesday,False,8,139.12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,4,Wednesday,False,8,139.12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,December,4,Wednesday,False,8,139.12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,4,Wednesday,False,8,139.12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,4,Wednesday,False,8,139.12


5. Exploratory Data Analysis

5.1 Sales Performance Analysis
    • KPIs
    • Revenue Distribution
    • Invoice Distribution

5.2 Product Performance Analysis
    • Top Products by Revenue
    • Top Products by Quantity
    • Most Frequently Purchased Products

5.3 Customer Purchasing Behaviour
    • Top Customers
    • Revenue per Customer
    • Orders per Customer

5.4 Geographic Analysis
    • Revenue by Country
    • Orders by Country

5.5 Time-Based Analysis
    • Monthly Revenue
    • Quarterly Revenue
    • Weekday Sales
    • Hourly Sales